Upload Datasets

In [10]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset (1).csv
Saving Twitter_Data.csv to Twitter_Data.csv


Import Libraries

In [11]:
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

Load Datasets

In [12]:
imdb = pd.read_csv("IMDB Dataset.csv")
twitter = pd.read_csv("Twitter_Data.csv")

print(imdb.head())
print(twitter.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
   2401  Borderlands  Positive  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

  im getting on borderlands and i will murder you all ,  
0  I am coming to the borders and I will kill you...     
1  im getting on borderlands and i will kill you ...     
2  im coming on borderlands and i will murder you...     
3  im getting on borderlands 2 and i will murder ...     
4  im getting into borderlands and i can murder y...     


Preprocess & Clean Text

In [13]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"@\w+", "", text)           # remove mentions
    text = re.sub(r"#\w+", "", text)           # remove hashtags
    text = re.sub(r"[^a-z\s]", "", text)       # keep only letters
    text = re.sub(r"\s+", " ", text).strip()
    return text

Prepare IMDb Dataset

In [ ]:
imdb['review'] = imdb['review'].apply(clean_text)

# Convert labels: positive=1, negative=0
imdb['sentiment'] = imdb['sentiment'].map({'positive': 1, 'negative': 0})

imdb = imdb[['review', 'sentiment']]

Prepare Twitter Dataset

Combine Datasets

In [19]:
data = pd.concat([imdb, twitter], ignore_index=True)
data = data.sample(frac=1).reset_index(drop=True)  # shuffle

print(data.head())
print(data['sentiment'].value_counts())

                                              review sentiment     2401  \
0                                                NaN       NaN  11672.0   
1                                                NaN       NaN   4676.0   
2  Trey Parker and Matt Stone, the creators of So...  positive      NaN   
3                                                NaN       NaN  10328.0   
4                                                NaN       NaN  10727.0   

                         Borderlands  Positive  \
0                            Verizon  Positive   
1                             Google  Negative   
2                                NaN       NaN   
3  PlayerUnknownsBattlegrounds(PUBG)   Neutral   
4             RedDeadRedemption(RDR)  Positive   

  im getting on borderlands and i will murder you all ,  
0  @optimum Why is someone in CT paying $100.00 l...     
1                              Why knew I quit using     
2                                                NaN     
3               

Train/Test Split

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Clean missing values
data = data.dropna(subset=['review', 'sentiment'])

# 2. Split data
X = data['review']
y = data['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# 5. Accuracy
print("Accuracy:", model.score(X_test_vec, y_test))

Accuracy: 0.8895


Train Model (Logistic Regression)

In [22]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)

LogisticRegression()

Evaluate Model (IMPORTANT for your supervisor)

In [23]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.8895

Report:
               precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4998
    positive       0.88      0.90      0.89      5002

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



Save Model (VERY IMPORTANT)

In [24]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

Download Files

In [25]:
files.download("model.pkl")
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>